In [9]:
# import libraries 


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# Added train_test_split here
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV 
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.ensemble import IsolationForest
# Added Pipeline here
from sklearn.pipeline import Pipeline 
import xgboost as xgb
import shap
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import joblib
import warnings
warnings.filterwarnings('ignore')


df = pd.read_csv('workflow_engineered.csv')
cat_cols = ['Process_Name', 'Task_Type', 'Priority_Level', 'Department', 'Approval_Level']
for col in cat_cols:
    df[col] = df[col].astype('category')



In [10]:
# Load models (or retrain – we'll retrain with best parameters)
feature_cols = ['Estimated_Time_Minutes', 'Employee_Workload', 
                'start_hour', 'start_dayofweek', 'start_month_num'] + cat_cols
X = df[feature_cols]
y_delay = df['Delay_Flag']
y_duration = df['Actual_Time_Minutes']
y_cost = df['Cost_Per_Task']

numeric_features = ['Estimated_Time_Minutes', 'Employee_Workload', 'start_hour', 'start_dayofweek', 'start_month_num']
categorical_features = cat_cols

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])


In [11]:
# 1. Hyperparameter tuning for Delay (XGBoost) 


X_train, X_test, y_train, y_test = train_test_split(X, y_delay, test_size=0.2, random_state=42, stratify=y_delay)

xgb_model = xgb.XGBClassifier(random_state=42, eval_metric='logloss')
param_grid = {
    'clf__n_estimators': [100, 200],
    'clf__max_depth': [3, 5, 7],
    'clf__learning_rate': [0.01, 0.1, 0.2],
    'clf__subsample': [0.8, 1.0]
}
pipeline = Pipeline([('prep', preprocessor), ('clf', xgb_model)])
grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)
print("Best params for Delay:", grid.best_params_)
print("Best F1 score:", grid.best_score_)

best_delay = grid.best_estimator_
y_pred = best_delay.predict(X_test)
print("\nTest set performance:")
print(classification_report(y_test, y_pred))
 


Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best params for Delay: {'clf__learning_rate': 0.01, 'clf__max_depth': 3, 'clf__n_estimators': 100, 'clf__subsample': 0.8}
Best F1 score: 0.9738323294164115

Test set performance:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        25
           1       0.95      1.00      0.97       475

    accuracy                           0.95       500
   macro avg       0.47      0.50      0.49       500
weighted avg       0.90      0.95      0.93       500



In [ ]:
# Feature Importance for Delay
xgb_imp = best_delay.named_steps['clf'].feature_importances_
cat_feature_names = best_delay.named_steps['prep'].named_transformers_['cat'].get_feature_names_out(categorical_features)
all_feature_names = numeric_features + list(cat_feature_names)
imp_df = pd.DataFrame({'feature': all_feature_names, 'importance': xgb_imp}).sort_values('importance', ascending=False)
print("Top 10 important features for delay prediction:\n", imp_df.head(10))

plt.figure(figsize=(10,6))
sns.barplot(x='importance', y='feature', data=imp_df.head(15))
plt.title('Top 15 Features – Delay Prediction')
plt.tight_layout()
plt.show()